### **Random forest MTL na embeddingach - zestaw 1 - Absorpcja

Wykorzystana reprezentacja: **embeddingi MoLFormer**

Lista endpointów:


1. Caco-2 (Wang)
2. Lipophilicity (AstraZeneca)
3. Solubility (AqSolDB)
4. HIA (Hou)
5. AMES Mutagenicity


In [ ]:
!pip install rdkit
!pip install pandas numpy scikit-learn -U
!pip install pytdc --no-dependencies
!pip install fuzzywuzzy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
data_folder = "/content/drive/MyDrive/data_splits"

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from tdc.single_pred import ADME
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_auc_score, accuracy_score, f1_score

In [ ]:
def load_embeddings(dataset_name, train_df, test_df):
    """Wczytuje CSV z embeddingami i dzieli na train/test po SMILES."""
    file_path = os.path.join(data_folder, f'{dataset_name}_MoLFormer_embeddings.csv')
    emb_df = pd.read_csv(file_path)

    train_smiles = set(train_df['Drug'].tolist())
    test_smiles  = set(test_df['Drug'].tolist())

    emb_train = emb_df[emb_df['Drug'].isin(train_smiles)]
    emb_test  = emb_df[emb_df['Drug'].isin(test_smiles)]

    feature_cols = [c for c in emb_df.columns if c.startswith('emb_')]

    X_train = emb_train[feature_cols].values
    y_train = emb_train['Y'].values
    X_test  = emb_test[feature_cols].values
    y_test  = emb_test['Y'].values

    return X_train, y_train, X_test, y_test, emb_train, emb_test

In [7]:
import pickle

def load_split_pickle(dataset_name):
    filepath = f"{data_folder}/{dataset_name}_split.pkl"

    with open(filepath, "rb") as f:
        split = pickle.load(f)

    return split["train"], split["test"]

In [ ]:
def print_metrics(metrics, task='classification', weight_loss_func_name=None):
    print(f"\n{'='*40}")
    if weight_loss_func_name:
        print(f"  Loss Weighting: {weight_loss_func_name}")
        print(f"{'='*40}")
    if task == 'classification':
        print(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}")
        print(f"  F1       : {metrics['test_metrics']['f1']:.4f}")
        print(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}")
    else:
        print(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}")
        print(f"  MAE      : {metrics['test_metrics']['mae']:.4f}")
        print(f"  R²       : {metrics['test_metrics']['r2']:.4f}")
    print(f"{'='*40}\n")


def save_metrics(metrics, dataset_name, filepath, task='classification', weight_loss_func_name=None, endpoint_group_name=None):
    with open(filepath, 'a') as f:
        f.write(f"\n{'='*40}\n")
        f.write(f"Endpoint    : {dataset_name}\n")
        if endpoint_group_name:
            f.write(f"Tasks       : {endpoint_group_name}\n")
        if weight_loss_func_name:
            f.write(f"Loss Weighting: {weight_loss_func_name}\n")
        f.write(f"{'='*40}\n")
        if task == 'classification':
            f.write(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}\n")
            f.write(f"  F1       : {metrics['test_metrics']['f1']:.4f}\n")
            f.write(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}\n")
        else:
            f.write(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}\n")
            f.write(f"  MAE      : {metrics['test_metrics']['mae']:.4f}\n")
            f.write(f"  R²       : {metrics['test_metrics']['r2']:.4f}\n")
        f.write(f"{'='*40}\n")

In [ ]:
import numpy as np
import pandas as pd

def prepare_mtl_data_final(df_list, task_names, featurizer):
    # 1. Zebranie wszystkich unikalnych struktur
    all_drugs = set()
    for df in df_list:
        valid = df['Drug'].dropna().astype(str).unique()
        all_drugs.update(valid)

    # 2. Walidacja cząsteczek przez RDKit przed stworzeniem master_list
    safe_master_list = []
    for drug in sorted(list(all_drugs)):
        mol = Chem.MolFromSmiles(drug)
        if mol: safe_master_list.append(drug)

    drug_to_idx = {drug: i for i, drug in enumerate(safe_master_list)}
    n_samples = len(safe_master_list)

    # 3. Generowanie X (tylko dla poprawnych cząsteczek)
    df_temp = pd.DataFrame({'Drug': safe_master_list})
    X_features, _ = featurizer(df_temp)

    # Sprawdzenie czy X zawiera NaN (zabezpieczenie przed błędami featurizera)
    if np.isnan(X_features).any():
        X_features = np.nan_to_num(X_features)

    # 4. Mapowanie etykiet y (z zachowaniem NaN tylko w etykietach!)
    y_dict = {}
    for df, task in zip(df_list, task_names):
        y_vec = np.full((n_samples, 1), np.nan, dtype=np.float32)
        # Tworzymy mapę {Drug: Wynik}
        mapping = dict(zip(df['Drug'].astype(str), df['Y']))

        for drug, val in mapping.items():
            if drug in drug_to_idx and not pd.isna(val):
                y_vec[drug_to_idx[drug]] = val
        y_dict[task] = y_vec

    return X_features, y_dict

In [10]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os

In [11]:
def train_hybrid_mtl_rf(X_train, y_train_dict, X_test, y_test_dict, reg_tasks, class_tasks):
    all_tasks = reg_tasks + class_tasks

    #tworzenie wspólnej macierzy
    Y_train_raw = np.hstack([y_train_dict[task] for task in all_tasks])
    Y_test_raw = np.hstack([y_test_dict[task] for task in all_tasks])

    Y_train_imputed = Y_train_raw.copy()
    Y_test_imputed = Y_test_raw.copy()

    print(">> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...")
    # Aby pozbyć się NaN, trenujemy szybkie modele bazowe.
    #Dla zadań regresyjnych używamy regresora, a dla klasyfikacyjnych klasyfikatora, wyciągając z niego prawdopodobieństwa, aby zachować ciągłość danych.
    for i, task in enumerate(all_tasks):
        known_train_idx = ~np.isnan(Y_train_raw[:, i])
        missing_train_idx = np.isnan(Y_train_raw[:, i])
        missing_test_idx = np.isnan(Y_test_raw[:, i])

        if task in reg_tasks:
            model_single = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
            model_single.fit(X_train[known_train_idx], Y_train_raw[known_train_idx, i])
            if np.any(missing_train_idx):
                Y_train_imputed[missing_train_idx, i] = model_single.predict(X_train[missing_train_idx])
            if np.any(missing_test_idx):
                Y_test_imputed[missing_test_idx, i] = model_single.predict(X_test[missing_test_idx])

        elif task in class_tasks:
            model_single = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1)
            model_single.fit(X_train[known_train_idx], Y_train_raw[known_train_idx, i].astype(int))
            if np.any(missing_train_idx):
                Y_train_imputed[missing_train_idx, i] = model_single.predict_proba(X_train[missing_train_idx])[:, 1]
            if np.any(missing_test_idx):
                Y_test_imputed[missing_test_idx, i] = model_single.predict_proba(X_test[missing_test_idx])[:, 1]

    print(">> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...")
    mtl_model = RandomForestRegressor(n_estimators=1000, max_depth=25, max_features="sqrt", random_state=42, n_jobs=-1)
    mtl_model.fit(X_train, Y_train_imputed)

    Y_pred = mtl_model.predict(X_test)

    # Budowanie słownika dopasowanego do funkcji print/save
    metrics_summary = {}
    for i, task in enumerate(all_tasks):
        known_test_idx = ~np.isnan(Y_test_raw[:, i])
        true_y = Y_test_raw[known_test_idx, i]
        pred_y = Y_pred[known_test_idx, i]

        if task in reg_tasks:
            metrics_summary[task] = {
                "task_type": "regression",  # Flaga pomocnicza dla pętli
                "test_metrics": {
                    "rmse": np.sqrt(mean_squared_error(true_y, pred_y)),
                    "mae": mean_absolute_error(true_y, pred_y),
                    "r2": r2_score(true_y, pred_y)
                }
            }
        elif task in class_tasks:
            pred_classes = (pred_y >= 0.5).astype(int)
            true_classes = true_y.astype(int)
            metrics_summary[task] = {
                "task_type": "classification",  # Flaga pomocnicza dla pętli
                "test_metrics": {
                    "accuracy": accuracy_score(true_classes, pred_classes),
                    "f1": f1_score(true_classes, pred_classes, zero_division=0),
                    "auroc": roc_auc_score(true_classes, pred_y)
                }
            }

    return mtl_model, metrics_summary

# Test1: Caco-2 (Wang) + Lipophilicity (Astra Zeneca)

In [ ]:
# 1. Konfiguracja
reg_tasks = ['Caco2_Wang', 'Lipophilicity_AstraZeneca']
class_tasks = []
all_tasks = reg_tasks + class_tasks
filepath = "mtl_results_absorpcja_embeddings_rf.txt"

# 2. Ładowanie oryginalnych podziałów pickle
train_caco2, test_caco2 = load_split_pickle('Caco2_Wang')
train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')

print(">> 1. Ładowanie embeddingów za pomocą Twojej funkcji load_embeddings...")
# Wywołujemy Twoją funkcję - ignorujemy czyste macierze (podłogi), bierzemy tabele z kolumną 'Drug'
_, _, _, _, emb_tr_caco2, emb_te_caco2 = load_embeddings('Caco2_Wang', train_caco2, test_caco2)
_, _, _, _, emb_tr_lipo, emb_te_lipo = load_embeddings('Lipophilicity_AstraZeneca', train_lipo, test_lipo)


df_train_list = [emb_tr_caco2, emb_tr_lipo]
df_test_list = [emb_te_caco2, emb_te_lipo]

print(">> 2. Tworzenie podręcznego słownika embeddingów...")
# Ustalamy nazwy kolumn z embeddingami
feature_cols = [c for c in emb_tr_caco2.columns if c.startswith('emb_')]

# Łączymy wszystkie załadowane dane w jeden słownik: {SMILES: wektor_liczb}
all_loaded_dfs = df_train_list + df_test_list
emb_dict = {}
for df in all_loaded_dfs:
    for drug, emb_vector in zip(df['Drug'].astype(str), df[feature_cols].values):
        emb_dict[drug] = emb_vector


featurizer = lambda df: (np.array([emb_dict.get(str(smiles), np.zeros(len(feature_cols))) for smiles in df['Drug']]), None)


print(">> 3. Przygotowywanie zintegrowanych zbiorów danych MTL...")
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, featurizer)
X_test_mtl, y_test_dict   = prepare_mtl_data_final(df_test_list, all_tasks, featurizer)

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

print("\n>>> RAPORTY KOŃCOWYCH METRYK <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]

    print(f"Endpoint: {task_name} ({task_type.upper()})")
    print_metrics(task_data, task=task_type)

    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="Caco2 + Lipophilicity"
    )

>> 1. Ładowanie embeddingów za pomocą Twojej funkcji load_embeddings...
>> 2. Tworzenie podręcznego słownika embeddingów...
>> 3. Przygotowywanie zintegrowanych zbiorów danych MTL...
>> Uruchamianie hybrydowego treningu Multi-Task Random Forest...
>> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...
>> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...

>>> RAPORTY KOŃCOWYCH METRYK <<<
Endpoint: Caco2_Wang (REGRESSION)

  RMSE     : 0.6313
  MAE      : 0.4957
  R²       : 0.3714

Endpoint: Lipophilicity_AstraZeneca (REGRESSION)

  RMSE     : 0.9724
  MAE      : 0.7717
  R²       : 0.3600



# Test2: Caco-2 (Wang) + Lipophilicity (Astra Zeneca) + Solubility (AqSolDB)

In [ ]:
# 1. Konfiguracja
reg_tasks = ['Caco2_Wang', 'Lipophilicity_AstraZeneca', 'Solubility_AqSolDB']
class_tasks = []
all_tasks = reg_tasks + class_tasks
filepath = "mtl_results_absorpcja_embeddings.txt"

# 2. Ładowanie oryginalnych podziałów pickle
train_caco2, test_caco2 = load_split_pickle('Caco2_Wang')
train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')
train_sol, test_sol = load_split_pickle('Solubility_AqSolDB')

print(">> 1. Ładowanie embeddingów za pomocą Twojej funkcji load_embeddings...")
_, _, _, _, emb_tr_caco2, emb_te_caco2 = load_embeddings('Caco2_Wang', train_caco2, test_caco2)
_, _, _, _, emb_tr_lipo, emb_te_lipo = load_embeddings('Lipophilicity_AstraZeneca', train_lipo, test_lipo)
_, _, _, _, emb_tr_sol, emb_te_sol = load_embeddings('Solubility_AqSolDB', train_sol, test_sol)


df_train_list = [emb_tr_caco2, emb_tr_lipo, emb_tr_sol]
df_test_list = [emb_te_caco2, emb_te_lipo, emb_te_sol]

print(">> 2. Tworzenie podręcznego słownika embeddingów...")
feature_cols = [c for c in emb_tr_caco2.columns if c.startswith('emb_')]

all_loaded_dfs = df_train_list + df_test_list
emb_dict = {}
for df in all_loaded_dfs:
    for drug, emb_vector in zip(df['Drug'].astype(str), df[feature_cols].values):
        emb_dict[drug] = emb_vector


featurizer = lambda df: (np.array([emb_dict.get(str(smiles), np.zeros(len(feature_cols))) for smiles in df['Drug']]), None)


print(">> 3. Przygotowywanie zintegrowanych zbiorów danych MTL...")
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, featurizer)
X_test_mtl, y_test_dict   = prepare_mtl_data_final(df_test_list, all_tasks, featurizer)

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

print("\n>>> RAPORTY KOŃCOWYCH METRYK <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]

    print(f"Endpoint: {task_name} ({task_type.upper()})")
    print_metrics(task_data, task=task_type)

    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="Caco2 + Lipophilicity + Solubility"
    )

>> 1. Ładowanie embeddingów za pomocą Twojej funkcji load_embeddings...
>> 2. Tworzenie podręcznego słownika embeddingów...
>> 3. Przygotowywanie zintegrowanych zbiorów danych MTL...


[23:45:16] WARNING: not removing hydrogen atom without neighbors
[23:45:16] WARNING: not removing hydrogen atom without neighbors
[23:45:16] WARNING: not removing hydrogen atom without neighbors
[23:45:16] WARNING: not removing hydrogen atom without neighbors
[23:45:16] WARNING: not removing hydrogen atom without neighbors
[23:45:16] WARNING: not removing hydrogen atom without neighbors
[23:45:16] WARNING: not removing hydrogen atom without neighbors
[23:45:17] WARNING: not removing hydrogen atom without neighbors
[23:45:17] WARNING: not removing hydrogen atom without neighbors
[23:45:17] WARNING: not removing hydrogen atom without neighbors
[23:45:17] WARNING: not removing hydrogen atom without neighbors
[23:45:17] WARNING: not removing hydrogen atom without neighbors
[23:45:17] WARNING: not removing hydrogen atom without neighbors
[23:45:17] WARNING: not removing hydrogen atom without neighbors
[23:45:17] WARNING: not removing hydrogen atom without neighbors
[23:45:17] WARNING: not r

>> Uruchamianie hybrydowego treningu Multi-Task Random Forest...
>> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...
>> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...

>>> RAPORTY KOŃCOWYCH METRYK <<<
Endpoint: Caco2_Wang (REGRESSION)

  RMSE     : 0.6546
  MAE      : 0.4996
  R²       : 0.3240

Endpoint: Lipophilicity_AstraZeneca (REGRESSION)

  RMSE     : 1.0260
  MAE      : 0.8238
  R²       : 0.2876

Endpoint: Solubility_AqSolDB (REGRESSION)

  RMSE     : 1.4324
  MAE      : 1.1104
  R²       : 0.6219



# Test3:  Caco-2 (Wang) + Lipophilicity (Astra Zeneca) + Solubility (AqSolDB) + HIA (Hou)

In [ ]:
# 1. Konfiguracja
reg_tasks = ['Caco2_Wang', 'Lipophilicity_AstraZeneca', 'Solubility_AqSolDB']
class_tasks = ['HIA_Hou']
all_tasks = reg_tasks + class_tasks
filepath = "mtl_results_absorpcja_embeddings.txt"

# 2. Ładowanie oryginalnych podziałów pickle
train_caco2, test_caco2 = load_split_pickle('Caco2_Wang')
train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')
train_sol, test_sol = load_split_pickle('Solubility_AqSolDB')
train_hia, test_hia = load_split_pickle('HIA_Hou')

print(">> 1. Ładowanie embeddingów za pomocą Twojej funkcji load_embeddings...")
_, _, _, _, emb_tr_caco2, emb_te_caco2 = load_embeddings('Caco2_Wang', train_caco2, test_caco2)
_, _, _, _, emb_tr_lipo, emb_te_lipo = load_embeddings('Lipophilicity_AstraZeneca', train_lipo, test_lipo)
_, _, _, _, emb_tr_sol, emb_te_sol = load_embeddings('Solubility_AqSolDB', train_sol, test_sol)
_, _, _, _, emb_tr_hia, emb_te_hia = load_embeddings('HIA_Hou', train_hia, test_hia)


df_train_list = [emb_tr_caco2, emb_tr_lipo, emb_tr_sol, emb_tr_hia]
df_test_list = [emb_te_caco2, emb_te_lipo, emb_te_sol, emb_te_hia]

print(">> 2. Tworzenie podręcznego słownika embeddingów...")
feature_cols = [c for c in emb_tr_caco2.columns if c.startswith('emb_')]

all_loaded_dfs = df_train_list + df_test_list
emb_dict = {}
for df in all_loaded_dfs:
    for drug, emb_vector in zip(df['Drug'].astype(str), df[feature_cols].values):
        emb_dict[drug] = emb_vector


featurizer = lambda df: (np.array([emb_dict.get(str(smiles), np.zeros(len(feature_cols))) for smiles in df['Drug']]), None)

print(">> 3. Przygotowywanie zintegrowanych zbiorów danych MTL...")
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, featurizer)
X_test_mtl, y_test_dict   = prepare_mtl_data_final(df_test_list, all_tasks, featurizer)

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

print("\n>>> RAPORTY KOŃCOWYCH METRYK <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]

    print(f"Endpoint: {task_name} ({task_type.upper()})")
    print_metrics(task_data, task=task_type)

    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="Caco2 + Lipophilicity + Solubility + HIA"
    )

>> 1. Ładowanie embeddingów za pomocą Twojej funkcji load_embeddings...
>> 2. Tworzenie podręcznego słownika embeddingów...
>> 3. Przygotowywanie zintegrowanych zbiorów danych MTL...


[23:49:13] WARNING: not removing hydrogen atom without neighbors
[23:49:13] WARNING: not removing hydrogen atom without neighbors
[23:49:13] WARNING: not removing hydrogen atom without neighbors
[23:49:13] WARNING: not removing hydrogen atom without neighbors
[23:49:13] WARNING: not removing hydrogen atom without neighbors
[23:49:13] WARNING: not removing hydrogen atom without neighbors
[23:49:13] WARNING: not removing hydrogen atom without neighbors
[23:49:13] WARNING: not removing hydrogen atom without neighbors
[23:49:13] WARNING: not removing hydrogen atom without neighbors
[23:49:14] WARNING: not removing hydrogen atom without neighbors
[23:49:14] WARNING: not removing hydrogen atom without neighbors
[23:49:14] WARNING: not removing hydrogen atom without neighbors
[23:49:14] WARNING: not removing hydrogen atom without neighbors
[23:49:14] WARNING: not removing hydrogen atom without neighbors
[23:49:14] WARNING: not removing hydrogen atom without neighbors
[23:49:14] WARNING: not r

>> Uruchamianie hybrydowego treningu Multi-Task Random Forest...
>> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...
>> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...

>>> RAPORTY KOŃCOWYCH METRYK <<<
Endpoint: Caco2_Wang (REGRESSION)

  RMSE     : 0.6540
  MAE      : 0.5003
  R²       : 0.3253

Endpoint: Lipophilicity_AstraZeneca (REGRESSION)

  RMSE     : 1.0258
  MAE      : 0.8241
  R²       : 0.2878

Endpoint: Solubility_AqSolDB (REGRESSION)

  RMSE     : 1.4349
  MAE      : 1.1137
  R²       : 0.6206

Endpoint: HIA_Hou (CLASSIFICATION)

  Accuracy : 0.8448
  F1       : 0.9151
  AUROC    : 0.9550



# Test4: Caco-2 (Wang) + HIA (Hou)

In [ ]:
# 1. Konfiguracja
reg_tasks = ['Caco2_Wang']
class_tasks = ['HIA_Hou']
all_tasks = reg_tasks + class_tasks
filepath = "mtl_results_absorpcja_embeddings.txt"

# 2. Ładowanie oryginalnych podziałów pickle
train_caco2, test_caco2 = load_split_pickle('Caco2_Wang')
train_hia, test_hia = load_split_pickle('HIA_Hou')

print(">> 1. Ładowanie embeddingów za pomocą Twojej funkcji load_embeddings...")
_, _, _, _, emb_tr_caco2, emb_te_caco2 = load_embeddings('Caco2_Wang', train_caco2, test_caco2)
_, _, _, _, emb_tr_hia, emb_te_hia = load_embeddings('HIA_Hou', train_hia, test_hia)


df_train_list = [emb_tr_caco2, emb_tr_hia]
df_test_list = [emb_te_caco2, emb_te_hia]

print(">> 2. Tworzenie podręcznego słownika embeddingów...")
feature_cols = [c for c in emb_tr_caco2.columns if c.startswith('emb_')]

all_loaded_dfs = df_train_list + df_test_list
emb_dict = {}
for df in all_loaded_dfs:
    for drug, emb_vector in zip(df['Drug'].astype(str), df[feature_cols].values):
        emb_dict[drug] = emb_vector


featurizer = lambda df: (np.array([emb_dict.get(str(smiles), np.zeros(len(feature_cols))) for smiles in df['Drug']]), None)

print(">> 3. Przygotowywanie zintegrowanych zbiorów danych MTL...")
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, featurizer)
X_test_mtl, y_test_dict   = prepare_mtl_data_final(df_test_list, all_tasks, featurizer)

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

print("\n>>> RAPORTY KOŃCOWYCH METRYK <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]

    print(f"Endpoint: {task_name} ({task_type.upper()})")
    print_metrics(task_data, task=task_type)

    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="Caco2 + HIA"
    )

>> 1. Ładowanie embeddingów za pomocą Twojej funkcji load_embeddings...
>> 2. Tworzenie podręcznego słownika embeddingów...
>> 3. Przygotowywanie zintegrowanych zbiorów danych MTL...
>> Uruchamianie hybrydowego treningu Multi-Task Random Forest...
>> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...
>> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...

>>> RAPORTY KOŃCOWYCH METRYK <<<
Endpoint: Caco2_Wang (REGRESSION)

  RMSE     : 0.5537
  MAE      : 0.4339
  R²       : 0.5164

Endpoint: HIA_Hou (CLASSIFICATION)

  Accuracy : 0.8621
  F1       : 0.9238
  AUROC    : 0.9848



# Test5: Lipophilicity (Astra Zeneca) + Solubility (AqSolDB)

In [ ]:
# 1. Konfiguracja
reg_tasks = [ 'Lipophilicity_AstraZeneca', 'Solubility_AqSolDB']
class_tasks = []
all_tasks = reg_tasks + class_tasks
filepath = "mtl_results_absorpcja_embeddings.txt"

# 2. Ładowanie oryginalnych podziałów pickle
train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')
train_sol, test_sol = load_split_pickle('Solubility_AqSolDB')

print(">> 1. Ładowanie embeddingów za pomocą Twojej funkcji load_embeddings...")
_, _, _, _, emb_tr_lipo, emb_te_lipo = load_embeddings('Lipophilicity_AstraZeneca', train_lipo, test_lipo)
_, _, _, _, emb_tr_sol, emb_te_sol = load_embeddings('Solubility_AqSolDB', train_sol, test_sol)


df_train_list = [emb_tr_lipo, emb_tr_sol]
df_test_list = [emb_te_lipo, emb_te_sol]

print(">> 2. Tworzenie podręcznego słownika embeddingów...")
feature_cols = [c for c in emb_tr_lipo.columns if c.startswith('emb_')]


all_loaded_dfs = df_train_list + df_test_list
emb_dict = {}
for df in all_loaded_dfs:
    for drug, emb_vector in zip(df['Drug'].astype(str), df[feature_cols].values):
        emb_dict[drug] = emb_vector


featurizer = lambda df: (np.array([emb_dict.get(str(smiles), np.zeros(len(feature_cols))) for smiles in df['Drug']]), None)

print(">> 3. Przygotowywanie zintegrowanych zbiorów danych MTL...")
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, featurizer)
X_test_mtl, y_test_dict   = prepare_mtl_data_final(df_test_list, all_tasks, featurizer)

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

print("\n>>> RAPORTY KOŃCOWYCH METRYK <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]

    print(f"Endpoint: {task_name} ({task_type.upper()})")
    print_metrics(task_data, task=task_type)

    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="Lipophilicity + Solubility"
    )

>> 1. Ładowanie embeddingów za pomocą Twojej funkcji load_embeddings...
>> 2. Tworzenie podręcznego słownika embeddingów...
>> 3. Przygotowywanie zintegrowanych zbiorów danych MTL...


[23:53:23] WARNING: not removing hydrogen atom without neighbors
[23:53:23] WARNING: not removing hydrogen atom without neighbors
[23:53:24] WARNING: not removing hydrogen atom without neighbors
[23:53:24] WARNING: not removing hydrogen atom without neighbors
[23:53:24] WARNING: not removing hydrogen atom without neighbors
[23:53:24] WARNING: not removing hydrogen atom without neighbors
[23:53:24] WARNING: not removing hydrogen atom without neighbors
[23:53:24] WARNING: not removing hydrogen atom without neighbors
[23:53:24] WARNING: not removing hydrogen atom without neighbors
[23:53:24] WARNING: not removing hydrogen atom without neighbors
[23:53:24] WARNING: not removing hydrogen atom without neighbors
[23:53:24] WARNING: not removing hydrogen atom without neighbors
[23:53:24] WARNING: not removing hydrogen atom without neighbors
[23:53:24] WARNING: not removing hydrogen atom without neighbors
[23:53:24] WARNING: not removing hydrogen atom without neighbors
[23:53:24] WARNING: not r

>> Uruchamianie hybrydowego treningu Multi-Task Random Forest...
>> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...
>> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...

>>> RAPORTY KOŃCOWYCH METRYK <<<
Endpoint: Lipophilicity_AstraZeneca (REGRESSION)

  RMSE     : 1.0246
  MAE      : 0.8212
  R²       : 0.2895

Endpoint: Solubility_AqSolDB (REGRESSION)

  RMSE     : 1.4272
  MAE      : 1.1070
  R²       : 0.6246



# Test6: Caco-2 (Wang) + AMES Mutagenicity

In [ ]:
# 1. Konfiguracja
reg_tasks = ['Caco2_Wang']
class_tasks = ['AMES']
all_tasks = reg_tasks + class_tasks
filepath = "mtl_results_absorpcja_embeddings.txt"

# 2. Ładowanie oryginalnych podziałów pickle
train_caco2, test_caco2 = load_split_pickle('Caco2_Wang')
train_ames, test_ames = load_split_pickle('AMES')

print(">> 1. Ładowanie embeddingów za pomocą Twojej funkcji load_embeddings...")
_, _, _, _, emb_tr_caco2, emb_te_caco2 = load_embeddings('Caco2_Wang', train_caco2, test_caco2)
_, _, _, _, emb_tr_ames, emb_te_ames = load_embeddings('AMES', train_ames, test_ames)


df_train_list = [emb_tr_caco2, emb_tr_ames]
df_test_list = [emb_te_caco2, emb_te_ames]

print(">> 2. Tworzenie podręcznego słownika embeddingów...")
feature_cols = [c for c in emb_tr_caco2.columns if c.startswith('emb_')]

all_loaded_dfs = df_train_list + df_test_list
emb_dict = {}
for df in all_loaded_dfs:
    for drug, emb_vector in zip(df['Drug'].astype(str), df[feature_cols].values):
        emb_dict[drug] = emb_vector


featurizer = lambda df: (np.array([emb_dict.get(str(smiles), np.zeros(len(feature_cols))) for smiles in df['Drug']]), None)


print(">> 3. Przygotowywanie zintegrowanych zbiorów danych MTL...")
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, featurizer)
X_test_mtl, y_test_dict   = prepare_mtl_data_final(df_test_list, all_tasks, featurizer)

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

print("\n>>> RAPORTY KOŃCOWYCH METRYK <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]

    print(f"Endpoint: {task_name} ({task_type.upper()})")
    print_metrics(task_data, task=task_type)

    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="Caco2 + AMES"
    )

>> 1. Ładowanie embeddingów za pomocą Twojej funkcji load_embeddings...
>> 2. Tworzenie podręcznego słownika embeddingów...
>> 3. Przygotowywanie zintegrowanych zbiorów danych MTL...
>> Uruchamianie hybrydowego treningu Multi-Task Random Forest...
>> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...
>> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...

>>> RAPORTY KOŃCOWYCH METRYK <<<
Endpoint: Caco2_Wang (REGRESSION)

  RMSE     : 0.5772
  MAE      : 0.4506
  R²       : 0.4744

Endpoint: AMES (CLASSIFICATION)

  Accuracy : 0.8095
  F1       : 0.8285
  AUROC    : 0.8742



# Test7: Caco-2 (Wang)+ Lipophilicity (Astra Zeneca)  + AMES Mutagenicity

In [ ]:
# 1. Konfiguracja
reg_tasks = ['Caco2_Wang', 'Lipophilicity_AstraZeneca']
class_tasks = ['AMES']
all_tasks = reg_tasks + class_tasks
filepath = "mtl_results_absorpcja_embeddings.txt"

# 2. Ładowanie oryginalnych podziałów pickle
train_caco2, test_caco2 = load_split_pickle('Caco2_Wang')
train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')
train_ames, test_ames = load_split_pickle('AMES')

print(">> 1. Ładowanie embeddingów za pomocą Twojej funkcji load_embeddings...")
_, _, _, _, emb_tr_caco2, emb_te_caco2 = load_embeddings('Caco2_Wang', train_caco2, test_caco2)
_, _, _, _, emb_tr_lipo, emb_te_lipo = load_embeddings('Lipophilicity_AstraZeneca', train_lipo, test_lipo)
_, _, _, _, emb_tr_ames, emb_te_ames = load_embeddings('AMES', train_ames, test_ames)


df_train_list = [emb_tr_caco2, emb_tr_lipo, emb_tr_ames]
df_test_list = [emb_te_caco2, emb_te_lipo, emb_te_ames]

print(">> 2. Tworzenie podręcznego słownika embeddingów...")
feature_cols = [c for c in emb_tr_caco2.columns if c.startswith('emb_')]

all_loaded_dfs = df_train_list + df_test_list
emb_dict = {}
for df in all_loaded_dfs:
    for drug, emb_vector in zip(df['Drug'].astype(str), df[feature_cols].values):
        emb_dict[drug] = emb_vector


featurizer = lambda df: (np.array([emb_dict.get(str(smiles), np.zeros(len(feature_cols))) for smiles in df['Drug']]), None)


print(">> 3. Przygotowywanie zintegrowanych zbiorów danych MTL...")
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, featurizer)
X_test_mtl, y_test_dict   = prepare_mtl_data_final(df_test_list, all_tasks, featurizer)

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

print("\n>>> RAPORTY KOŃCOWYCH METRYK <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]

    print(f"Endpoint: {task_name} ({task_type.upper()})")
    print_metrics(task_data, task=task_type)

    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="Caco2 + Lipophilicity + AMES"
    )

>> 1. Ładowanie embeddingów za pomocą Twojej funkcji load_embeddings...
>> 2. Tworzenie podręcznego słownika embeddingów...
>> 3. Przygotowywanie zintegrowanych zbiorów danych MTL...
>> Uruchamianie hybrydowego treningu Multi-Task Random Forest...
>> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...
>> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...

>>> RAPORTY KOŃCOWYCH METRYK <<<
Endpoint: Caco2_Wang (REGRESSION)

  RMSE     : 0.6252
  MAE      : 0.4843
  R²       : 0.3834

Endpoint: Lipophilicity_AstraZeneca (REGRESSION)

  RMSE     : 0.9970
  MAE      : 0.7940
  R²       : 0.3272

Endpoint: AMES (CLASSIFICATION)

  Accuracy : 0.7999
  F1       : 0.8159
  AUROC    : 0.8707



# Test 8: Caco-2 (Wang) + Lipophilicity (Astra Zeneca) + Solubility (AqSolDB) + HIA (Hou)  + AMES

In [ ]:
# 1. Konfiguracja
reg_tasks = ['Caco2_Wang', 'Lipophilicity_AstraZeneca', 'Solubility_AqSolDB']
class_tasks = ['HIA_Hou', 'AMES']
all_tasks = reg_tasks + class_tasks
filepath = "mtl_results_absorpcja_embeddings.txt"

# 2. Ładowanie oryginalnych podziałów pickle
train_caco2, test_caco2 = load_split_pickle('Caco2_Wang')
train_lipo, test_lipo = load_split_pickle('Lipophilicity_AstraZeneca')
train_sol, test_sol = load_split_pickle('Solubility_AqSolDB')
train_hia, test_hia = load_split_pickle('HIA_Hou')
train_ames, test_ames = load_split_pickle('AMES')

print(">> 1. Ładowanie embeddingów za pomocą Twojej funkcji load_embeddings...")
_, _, _, _, emb_tr_caco2, emb_te_caco2 = load_embeddings('Caco2_Wang', train_caco2, test_caco2)
_, _, _, _, emb_tr_lipo, emb_te_lipo = load_embeddings('Lipophilicity_AstraZeneca', train_lipo, test_lipo)
_, _, _, _, emb_tr_sol, emb_te_sol = load_embeddings('Solubility_AqSolDB', train_sol, test_sol)
_, _, _, _, emb_tr_hia, emb_te_hia = load_embeddings('HIA_Hou', train_hia, test_hia)
_, _, _, _, emb_tr_ames, emb_te_ames = load_embeddings('AMES', train_ames, test_ames)


df_train_list = [emb_tr_caco2, emb_tr_lipo, emb_tr_sol, emb_tr_hia, emb_tr_ames]
df_test_list = [emb_te_caco2, emb_te_lipo, emb_te_sol, emb_te_hia, emb_te_ames]

print(">> 2. Tworzenie podręcznego słownika embeddingów...")
feature_cols = [c for c in emb_tr_caco2.columns if c.startswith('emb_')]


all_loaded_dfs = df_train_list + df_test_list
emb_dict = {}
for df in all_loaded_dfs:
    for drug, emb_vector in zip(df['Drug'].astype(str), df[feature_cols].values):
        emb_dict[drug] = emb_vector


featurizer = lambda df: (np.array([emb_dict.get(str(smiles), np.zeros(len(feature_cols))) for smiles in df['Drug']]), None)


print(">> 3. Przygotowywanie zintegrowanych zbiorów danych MTL...")
X_train_mtl, y_train_dict = prepare_mtl_data_final(df_train_list, all_tasks, featurizer)
X_test_mtl, y_test_dict   = prepare_mtl_data_final(df_test_list, all_tasks, featurizer)

print(">> Uruchamianie hybrydowego treningu Multi-Task Random Forest...")
mtl_model, metrics = train_hybrid_mtl_rf(
    X_train_mtl, y_train_dict,
    X_test_mtl, y_test_dict,
    reg_tasks, class_tasks
)

print("\n>>> RAPORTY KOŃCOWYCH METRYK <<<")
for task_name, task_data in metrics.items():
    task_type = task_data["task_type"]

    print(f"Endpoint: {task_name} ({task_type.upper()})")
    print_metrics(task_data, task=task_type)

    save_metrics(
        metrics=task_data,
        dataset_name=task_name,
        filepath=filepath,
        task=task_type,
        endpoint_group_name="Caco2 + Lipophilicity + Solubility + HIA + AMES"
    )

>> 1. Ładowanie embeddingów za pomocą Twojej funkcji load_embeddings...
>> 2. Tworzenie podręcznego słownika embeddingów...
>> 3. Przygotowywanie zintegrowanych zbiorów danych MTL...


[00:01:46] WARNING: not removing hydrogen atom without neighbors
[00:01:46] WARNING: not removing hydrogen atom without neighbors
[00:01:46] WARNING: not removing hydrogen atom without neighbors
[00:01:46] WARNING: not removing hydrogen atom without neighbors
[00:01:47] WARNING: not removing hydrogen atom without neighbors
[00:01:47] WARNING: not removing hydrogen atom without neighbors
[00:01:47] WARNING: not removing hydrogen atom without neighbors
[00:01:47] WARNING: not removing hydrogen atom without neighbors
[00:01:47] WARNING: not removing hydrogen atom without neighbors
[00:01:47] WARNING: not removing hydrogen atom without neighbors
[00:01:47] WARNING: not removing hydrogen atom without neighbors
[00:01:47] WARNING: not removing hydrogen atom without neighbors
[00:01:47] WARNING: not removing hydrogen atom without neighbors
[00:01:47] WARNING: not removing hydrogen atom without neighbors
[00:01:47] WARNING: not removing hydrogen atom without neighbors
[00:01:47] WARNING: not r

>> Uruchamianie hybrydowego treningu Multi-Task Random Forest...
>> Krok 1: Imputacja brakujących wartości (Pseudo-labeling)...
>> Krok 2: Uruchamianie treningu wspólnego modelu Multi-Task Random Forest...

>>> RAPORTY KOŃCOWYCH METRYK <<<
Endpoint: Caco2_Wang (REGRESSION)

  RMSE     : 0.6512
  MAE      : 0.4993
  R²       : 0.3311

Endpoint: Lipophilicity_AstraZeneca (REGRESSION)

  RMSE     : 1.0342
  MAE      : 0.8316
  R²       : 0.2761

Endpoint: Solubility_AqSolDB (REGRESSION)

  RMSE     : 1.4523
  MAE      : 1.1273
  R²       : 0.6113

Endpoint: HIA_Hou (CLASSIFICATION)

  Accuracy : 0.8448
  F1       : 0.9151
  AUROC    : 0.9615

Endpoint: AMES (CLASSIFICATION)

  Accuracy : 0.7785
  F1       : 0.7821
  AUROC    : 0.8633

